This code was mostly adapted from the Qwen-VL GitHub repository: https://github.com/QwenLM/Qwen-VL/tree/master

In [ ]:
!pip install tiktoken
!pip install fastapi nest-asyncio pyngrok uvicorn

In [ ]:
!pip install einops transformers_stream_generator

In [ ]:
!pip install accelerate

In [ ]:
!pip install gekko

In [ ]:
!pip install optimum
!pip install transformers==4.32.0 accelerate tiktoken einops transformers_stream_generator==0.0.4 scipy torchvision pillow tensorboard matplotlib

In [ ]:
!git clone https://github.com/JustinLin610/AutoGPTQ.git
%cd AutoGPTQ
!pip install -v .

In [ ]:
!pip install python-multipart

In [ ]:
# The historical wildcard deletion command was removed from this copy.


The following lines have been taken from the model's repository on github (specifically, from the tutorial.md file under "Initializing the Qwen_VL chat model): https://github.com/QwenLM/Qwen-VL/blob/master/TUTORIAL.md

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
import nest_asyncio
from pyngrok import ngrok
import uvicorn
import os

torch.manual_seed(1234)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen-VL-Chat-Int4", trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen-VL-Chat-Int4", device_map="cuda", trust_remote_code=True).eval()


Dynamic prompt and input handling. <br>
This code was adapted from this file in the Qwen-VL repository: https://github.com/QwenLM/Qwen-VL/blob/master/openai_api.py

In [ ]:
from fastapi import FastAPI, File, UploadFile, Form, HTTPException
from fastapi.responses import JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from pathlib import Path
from tempfile import TemporaryDirectory

# Adapted from the upstream Qwen-VL API example cited in the preceding cell.
# The original prototype used ChatGPT assistance for upload handling.
app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"]
)

@app.post("/uploadfile/")
async def create_upload_file(
    file: UploadFile = File(...),
    aud: str = Form("No Audio"),
    img: str = Form("No Image"),
):
    try:
        if aud in ["No Audio", "nothing"] and img == "nothing":
            message = "A P300 spike was detected but no significant environmental factors were noted."
            return JSONResponse(content={"response": message})

        prompt = (
            "Help me! Summarise the situation in a single sentence, and then generate another sentence where you give me a piece of helpful advice."
            "Talk to me personally, the user is driving. Structure your response with 'Situation' and 'Advice'. "
            f"The detected audio category is '{aud}' and the detected image category is '{img}'."
        )
        # Temporary uploads are removed after success or failure, including model errors.
        with TemporaryDirectory(prefix="bci_upload_") as directory:
            image_path = Path(directory) / "input_image.jpg"
            image_path.write_bytes(await file.read())
            query = tokenizer.from_list_format([
                {"image": str(image_path)},
                {"text": prompt},
            ])
            response, _ = model.chat(tokenizer, query=query, history=None)
        return JSONResponse(content={"response": response})
    except Exception as exc:
        raise HTTPException(status_code=500, detail="Model inference failed.") from exc


In [ ]:
import os
import nest_asyncio
from pyngrok import ngrok
import uvicorn

auth_token = os.environ.get("NGROK_AUTHTOKEN")
if not auth_token:
    try:
        from google.colab import userdata
        auth_token = userdata.get("NGROK_AUTHTOKEN")
    except Exception:
        auth_token = None
if not auth_token:
    raise RuntimeError("Set NGROK_AUTHTOKEN in the environment or the Colab secret store.")

ngrok.set_auth_token(auth_token)
ngrok_tunnel = ngrok.connect(8000)
print("Public URL:", ngrok_tunnel.public_url)
nest_asyncio.apply()
uvicorn.run(app, host="127.0.0.1", port=8000, access_log=False)
